<a href="https://colab.research.google.com/github/Brandon9010/Brandon9010/blob/main/Pool_Inventory_Forecasting/01_generate_synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Inventory Forecasting

##1. Create synthetic data

Import Libraries and set seed

In [2]:
#Import libraries
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)

Load Product Structure

In [3]:
#Initialize product list
real_products = [
    # Weekly Maintenance (High Velocity)
    ("Pool Shock", "1lb", 50, 185.00),
    ("Pool Shock", "5lb", 6, 85.00),
    ("Pool Shock", "17lb", 1, 45.00),
    ("Pool Shock", "25lb", 1, 65.00),
    ("Liquid Shock", "1gal", 4, 18.00),
    ("Chlorine Tablets", "1lb", 12, 45.00),
    ("Chlorine Tablets", "11lb", 4, 130.00),
    ("Chlorine Tablets", "21lb", 1, 65.00),
    ("Chlorine Tablets", "25lb", 1, 75.00),
    ("Basic Algaecide", "32oz", 12, 90.00),
    ("Copper Algaecide", "32oz", 12, 110.00),
    ("Mustard Algaecide", "32oz", 12, 125.00),
    ("Black Algaecide", "32oz", 12, 140.00),
    # Seasonal Balancers (Low Velocity)
    ("pH Up", "2lb", 6, 20.00),
    ("pH Up", "5lb", 6, 35.00),
    ("pH Down", "2lb", 6, 22.00),
    ("pH Down", "5lb", 6, 38.00),
    ("Muriatic Acid", "1gal", 4, 25.00),
    ("Stabilizer/Conditioner", "1lb", 12, 65.00),
    ("Stabilizer/Conditioner", "5lb", 6, 85.00),
    ("Alkalinity Increaser", "5lb", 6, 45.00),
    ("Alkalinity Increaser", "11lb", 4, 60.00),
    ("Alkalinity Increaser", "25lb", 1, 35.00),
    ("Calcium Hardness Increaser", "7lb", 6, 55.00),
    ("Calcium Hardness Increaser", "11lb", 6, 80.00),
    ("Calcium Hardness Increaser", "20lb", 1, 30.00),
    ("Flocculant", "2lb", 12, 75.00)
]

distributors = ['BioGuard Logistics', 'PoolCorp Supply', 'AquaClear Wholesalers', 'Chemical Direct']
brands = ['Chlorox', 'AquaChem', 'BioGuard', 'HTH', 'InTheSwim']

prod_data = []
for idx, (name, size, case_qty, est_cost) in enumerate(real_products):
    brand = random.choice(brands)
    dist = random.choice(distributors)

    sku_prefix = "".join([word[0] for word in name.split()])[:3].upper()
    sku = f"{brand[:3].upper()}-{sku_prefix}-{size.upper()}"

    # Add minor noise to the base cost
    base_cost = round(est_cost * random.uniform(0.95, 1.05), 2)

    # THE UPDATE: ~100% Markup (Multiplier between 1.9x and 2.1x)
    retail_price = round(base_cost * random.uniform(1.9, 2.1), 2)

    cost_per_unit = round(base_cost / case_qty, 2)
    retail_per_unit = round(retail_price / case_qty, 2)

    prod_data.append([sku, f"{brand} {name} ({size})", name, brand, dist, size, case_qty, base_cost, cost_per_unit, retail_per_unit])

df_products = pd.DataFrame(prod_data, columns=['SKU', 'Product_Name', 'Category', 'Brand', 'Distributor', 'Unit_Size', 'Case_Qty', 'Case_Cost', 'Unit_Cost', 'Unit_Retail'])


Generate Historical Sales Data

In [4]:
sales_data = []
start_date = datetime(2021, 1, 1)
end_date = datetime(2023, 12, 31)
date_list = [start_date + timedelta(days=x) for x in range((end_date-start_date).days + 1)]

# Establish Velocity Weights based on domain knowledge
weights = []
for _, row in df_products.iterrows():
    cat = row['Category']
    if "Shock" in cat or "Tablet" in cat or "Algaecide" in cat:
        weights.append(15)  # Tier 1: Weekly Maintenance (Huge likelihood of sale)
    elif "Acid" in cat or "Liquid" in cat:
        weights.append(5)   # Tier 2: Regular Additives
    else:
        weights.append(1)   # Tier 3: Seasonal Balancers (Rarely bought after June)

# Normalize weights so they add up to 1.0
weights = np.array(weights) / sum(weights)

for date in date_list:
    month = date.month
    seasonality_multiplier = max(0.05, np.exp(-0.15 * (month - 7)**2))

    num_transactions = int(np.random.normal(45, 10) * seasonality_multiplier)
    num_transactions = max(2, num_transactions)

    for _ in range(num_transactions):
        # Pick a product using our new Velocity Weights
        sku = np.random.choice(df_products['SKU'].tolist(), p=weights)

        # Determine quantity (customers usually buy 1-4 individual units at a retail store)
        qty_sold = np.random.choice([1, 2, 3, 4], p=[0.7, 0.15, 0.1, 0.05])

        txn_time = date + timedelta(hours=random.randint(8, 18), minutes=random.randint(0, 59))
        sales_data.append([txn_time, sku, qty_sold])

df_sales = pd.DataFrame(sales_data, columns=['Transaction_Date', 'SKU', 'Unit_Qty_Sold'])

Export to CSV

In [5]:
df_products.to_csv('pool_products_master.csv', index=False)
df_sales.to_csv('pool_sales_history.csv', index=False)

print("Domain-Optimized Supply Chain Data Generated!")
print(f"Master Inventory Table: {df_products.shape[0]} unique SKUs.")
print(f"Sales Table: {df_sales.shape[0]} historical transactions.")

# Quick validation check to ensure the markup worked:
print("\n--- Margin Validation Check ---")
sample = df_products[['Product_Name', 'Unit_Cost', 'Unit_Retail']].head(3).copy()
sample['Markup_%'] = ((sample['Unit_Retail'] - sample['Unit_Cost']) / sample['Unit_Cost'] * 100).round(1)
print(sample)

Domain-Optimized Supply Chain Data Generated!
Master Inventory Table: 27 unique SKUs.
Sales Table: 18671 historical transactions.

--- Margin Validation Check ---
                 Product_Name  Unit_Cost  Unit_Retail  Markup_%
0   BioGuard Pool Shock (1lb)       3.60         6.97      93.6
1   BioGuard Pool Shock (5lb)      13.87        29.01     109.2
2  BioGuard Pool Shock (17lb)      47.15        97.11     106.0
